# Exercise 5 · Let Codex do the Git

You will ask Codex to make one change, commit it with your message and push it. Then you check what it did.

**Goal:** Have Codex commit one change with an exact message and push it, approve each step it asks about, and check the result yourself.

**Why it matters:** A coding agent can run Git for you. You still decide what it may send to GitHub, and you check what arrived.

**Before you start:**
- You did exercises 1 and 3 today: you have your own copy on GitHub, and `git push` works in your terminal.
- Step 2 clones your copy into this exercise's `practice/starter` with one command: `git clone <your link> starter`.
- Codex is installed and signed in. You know how to start it and leave it with `/exit`.

**What the check measures:** your newest commit adds `CHECKLIST.md` with the message `Add a challenge checklist`, and GitHub has that commit.

**Done when:** the last cell prints `Done:`.

## The situation

You want a checklist file in your copy. This time Codex makes the file, the commit and the push. You approve what goes online. The checklist is made up.

## Step 1 · Write your link

Run the cell with your link from exercise 1. It makes `practice/` and prints the clone command for Step 2. It does not go online.

Expect a line that starts with `git clone https://github.com/`.

In [ ]:
# Step 1: paste the link you handed in for exercise 1 between the quotes, then run this cell
MY_REPO = ""

import re
from pathlib import Path

PRACTICE = Path("practice")
PRACTICE.mkdir(exist_ok=True)
found = re.search(r"github\.com/([\w.-]+)/([\w.-]+)", MY_REPO)
if (PRACTICE / "starter").exists():
    print("Kept practice/starter: it is already there. Go to Step 3.")
elif not found:
    print("MY_REPO is empty or not a GitHub link. Paste your link from exercise 1, then run this cell again.")
else:
    print("In Step 2, clone your copy with this line:")
    print(f"git clone https://github.com/{found[1]}/{found[2].removesuffix('.git')}.git starter")

## Step 2 · Clone your copy

Choose **Terminal > New Terminal**. Type each line, then press Enter. For the third line, paste the line that Step 1 printed:

```sh
source .venv/bin/activate
cd week-02/day-02/exercise-05-codex-does-the-git/practice
git clone https://github.com/<your-account>/<name>.git starter
cd starter
```

Expect `Cloning into 'starter'...`.

### Learn: Codex asks before it goes online

In a Git folder, Codex starts in Auto. It edits files in the folder without asking. "Codex asks for approval to edit files outside the workspace or to run commands that require network access" ([Agent approvals & security](https://learn.chatgpt.com/docs/agent-approvals-security)). `git push` needs the network. `git commit` may also need approval, because Codex keeps `.git` read-only ([Advanced configuration](https://learn.chatgpt.com/docs/config-file/config-advanced)).

## Step 3 · Ask Codex to commit and push

```sh
codex --no-daemon
```

If your Codex does not know `--no-daemon`, leave it out. If Codex asks "Trust this folder?", choose `1. Trust and continue`. Then type this request and press Enter:

```text
Create CHECKLIST.md with one line: - [ ] Read CHALLENGE.md. Commit only that file with the message "Add a challenge checklist", then push it to origin main. Done when git status says the branch is up to date with 'origin/main'.
```

Expect Codex to make the file, then stop and ask before `git push`, and maybe before `git commit`.

## Step 4 · Read each request, then approve it once

Approve a request only if it runs the command you asked for. "Choose the narrowest scope that lets the task continue" ([Sandbox](https://learn.chatgpt.com/docs/sandboxing)): approve once, not for the session.

Expect Codex to say the push worked. If it says the push failed, type `/exit` and run `git push` yourself.

<details><summary>Show the answer: which requests do you approve?</summary>

Approve `git add CHECKLIST.md`, `git commit -m "Add a challenge checklist"` and `git push origin main`, or the same commands written another way. Decline a push with `--force`, or a commit of other files. Why: you asked for one file, one commit and one push.

</details>

## Step 5 · Check what Codex did

Type `/exit`. Then run:

```sh
git log --oneline -2
git status
```

Expect `Add a challenge checklist` at the top. Expect `Your branch is up to date with 'origin/main'.` Then open your copy on github.com: expect `CHECKLIST.md` in the file list.

## Start over (only if you need it)

If the commit is wrong and not pushed, run `git reset --hard origin/main`: it drops your unpushed commits. Then go to Step 3. If it is pushed, ask Codex for one more commit with the right change and message.

## Check your work

Run the cell. Expect three `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do. The last check asks GitHub, so it needs the internet.

In [ ]:
# Check your work: read the newest commit in practice/starter, then compare it with GitHub (a public read)
import os
import re
import subprocess
from pathlib import Path

CLONE = Path("practice") / "starter"
MESSAGE = "Add a challenge checklist"
NO_PROMPT = dict(os.environ, GIT_TERMINAL_PROMPT="0")


def git(*args) -> str:
    """Return git's output in practice/starter, or "" if the command fails."""
    try:
        done = subprocess.run(["git", "-C", str(CLONE), *args], capture_output=True, text=True,
                              timeout=60, env=NO_PROMPT)
    except (OSError, subprocess.TimeoutExpired):
        return ""
    return done.stdout.strip() if done.returncode == 0 else ""


def first_problem() -> str:
    """Print ok for each check that passes; return the first problem, or ""."""
    if not (CLONE / ".git").exists():
        return "practice/starter is missing. Clone your copy into it (Step 2)."
    owner = re.search(r"github\.com[/:]([\w.-]+)/", git("config", "--get", "remote.origin.url"))
    if not owner or owner[1].lower() == "gitrolltraining":
        return "practice/starter is not a clone of your copy on github.com. Clone your own link (Step 2)."
    print(f"ok: practice/starter is a clone of your copy ({owner[1]})")
    subject = git("log", "-1", "--format=%s")
    if subject != MESSAGE:
        return f"your newest commit is {subject!r}, not {MESSAGE!r}. Ask Codex again (Step 3)."
    if "CHECKLIST.md" not in git("diff-tree", "--no-commit-id", "--name-only", "-r", "HEAD").splitlines():
        return "your newest commit does not add CHECKLIST.md. Ask Codex again (Step 3)."
    print(f"ok: your newest commit is {MESSAGE!r} and adds CHECKLIST.md")
    remote = git("ls-remote", "origin", "HEAD").split()
    if not remote:
        return "Git could not read your copy on GitHub. Check your internet connection, then run this cell again."
    if remote[0] != git("rev-parse", "HEAD"):
        return "GitHub does not have your newest commit. Approve Codex's push (Step 4), or run git push yourself."
    print("ok: GitHub has your newest commit")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Do Step 2, then run this cell."
print(f"Not yet: {problem}" if problem else
      "Done: Codex committed and pushed the change you asked for, and you checked it yourself.")

## What you learned

- Codex can run `git add`, `git commit` and `git push` for you. It asks before it goes online.
- You approve each request once, then check the result with `git log`, `git status` and github.com.

**Reflect:** Which Git command would you never approve for an agent without reading it twice? Why?